# Geospatial foundation model embeddings

Geospatial foundation models are neural network models trained via self-supervision using large and diverse datasets of remote sensing images and other geospatial data. Due to training via self-supervision, they do not need a pre-labelled dataset for training which are scarce relative to the volume of satellite image archives. This means that geospatial foundation models can be trained with data covering a large geographic extent and a range of modalities (e.g. multispectral, LiDAR, elevation models, SAR, weather, text etc.). 

A trained geospatial foundation model can take in a range of geospatial datasets (e.g. a time-series of Sentinel-2 satellite images, a digital elevation model, and historical weather data) and convert this data into a numeric representation of the geographic context and characteristics of a location. This numeric representation is called an embedding. The embedding can be used in downstream analysis tasks such as image segmentation or classification, clustering, change detection or similarity searches. 

Some benefits of using embeddings from geospatial foundation models for predictive mapping are:

* Often, improved performance on segmentation or classification tasks (but this always needs checking!).
* Less labelled training data to develop task-specific machine learning models.
* Reduced data pre-processing and feature engineering efforts to leverage large volumes of diverse geospatial data.
* Adaptability and generalise to a range of mapping tasks and contexts. 

As the embeddings are information rich representations of a location generated from a geospatial foundation model that has already learnt from large volumes data, you can often fit simple models (i.e a linear regression) on the embeddings and achieve good predictive performance. The assumption here is that the embeddings are generated from pre-trained foundation models that have learnt how transform input data (e.g. satellite images) into useful representations, and, therefore, you do not need to train a model to do this. All you need to do is fit a linear model on top of the embeddings. The pitch for embeddings is that they've saved you from pre-processing large amounts of geospatial data, developing complex models and possibly having to collect more labelled data for model training. 

In this lab, you will explore how you can use the embeddings from a geospatial foundation model as predictor variables in a supervised learning task to develop models that predict key urban microclimate variables: % building cover and % tree cover within a grid cell (plan area fractions) and the mean height of buildings and trees (roughness elements). You will be working with <a href="https://deepmind.google/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail/" target="_blank">Google's Satellite Embeddings from the AlphaEarth Foundation model</a>. 

This <a href="https://www.spatialedge.co/p/google-deepminds-new-geospatial-model" target="_blank">blog</a> provides an introduction to geospatial foundation models and the AlphaEarth model.

You can watch the following video as introduction to the AlphaEarth model:

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("1oiyyIloqxo")

## Tasks

You will compare models trained to predict urban microclimate variables from Google's Satellite Embeddings to specialist models trained directly on remote sensing data inputs. For each of the following target variables:

* mean building height (m)
* mean tree height (m)
* % tree cover
* % building cover

You should develop this notebook to address the following tasks and questions:

* fit a linear model using Google's AlphaEarth Satellite Embeddings and a linear model using remote sensing data inputs (try with either Landsat, Sentinel-2, or both). 
* fit a more complex model (e.g. a small deep neural network - example provided below) on both the Satellite Embeddings and the remote sensing data inputs. 
* how does a linear fit on embeddings compare to more complex models fitted on "raw" remote sensing data?
* compare and contrast the performance of the models trained with embeddings versus "raw" remote sensing data. 
* which input dataset and model would you use to generate Perth-wide maps of these variables? Consider both model performance, costs of data processing for deployment, and interpretability. 

### Datasets

The dataset used here comprises 100 m x 100 m grid cells distributed in clusters across Perth. In each grid cell the area of building footprints were extracted from Overture Maps, the area of tree cover was extracted from Urban Monitor aerial images, and the height of buildings and trees were extracted from Urban Monitor surface models. These variables were used to create the following target variables per grid cell:

* mean building height (m)
* mean tree height (m)
* % tree cover
* % building cover

Our task is to develop machine learning models to predict these targets with the following predictors:

**Landsat summer composite (12)**

Pattern **`landsat_b{1-6}_{mean,std}`** — per-cell mean and SD rescaled to surface reflectance 0-1 (`DN * 2.75e-5 - 0.2`) with Collection 2 fill pixels dropped.

- `b1` blue · `b2` green · `b3` red · `b4` NIR · `b5` SWIR-1 · `b6` SWIR-2

**Sentinel-2 monthly reflectance (72)**

Pattern **`s2_{YYYYMM}_{band}_{mean,std}`** — months `201912`, `202001`, `202002` (summer 2019/20) x 12 bands.

Cloud-masked monthly median composite of `COPERNICUS/S2_SR_HARMONIZED` (s2cloudless probability < 40%), in **reflectance units 0-1**, reduced at 10 m.

- `B1` coastal aerosol · `B2` blue · `B3` green · `B4` red
- `B5` / `B6` / `B7` red-edge 1 / 2 / 3
- `B8` NIR · `B8A` narrow NIR · `B9` water vapour
- `B11` SWIR-1 · `B12` SWIR-2

`_mean` is the cell average; `_std` is the within-cell SD, i.e. spectral texture at 10 m.

**Sentinel-2 spectral indices (24)**

Pattern **`s2_{YYYYMM}_{index}_{mean,std}`** — same 3 months x 4 indices. Computed per scene *after* cloud masking and *then* median-composited, so these are median indices, not indices of median bands.

| Index | Formula | Meaning | Summer mean |
|---|---|---|---|
| `NDVI` | (B8-B4)/(B8+B4) | Greenness / vigour | ~0.28 |
| `NDRE` | (B8-B5)/(B8+B5) | Red-edge; chlorophyll and canopy stress | ~0.17 |
| `NDBI` | (B11-B8)/(B11+B8) | Built-up / impervious | ~0.03 |
| `SAVI` | (B8-B4)/(B8+B4+0.5) x 1.5 | Soil-adjusted greenness; damps bare-soil background | ~0.18 |

**Leaf area index (6)**

Pattern **`lai_{YYYYMM}_{mean,std}`** — 3 months. LEAF toolbox SL2P neural-net retrieval on Sentinel-2, monthly median composite. Units m² leaf per m² ground, range 0-2.94.

> *The only group with missing data: 62 nulls* — cells where no good-quality retrieval passed the QC mask. Every other column is complete.

**AlphaEarth satellite embeddings (64)**

Pattern **`emb_2020_A{00-63}_mean`** — per-cell mean of Google's `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` 2020 image: a 64-dimensional learned representation of each 10 m pixel's whole-year satellite record.


## Setup

### Load data

In [ ]:
import os
import subprocess

if "data-geoml" not in os.listdir(os.getcwd()):
    subprocess.run('wget "https://github.com/envt-5566/data/raw/main/data-geoml-2026.zip"', shell=True, capture_output=True, text=True)
    subprocess.run('unzip "data-geoml-2026.zip"', shell=True, capture_output=True, text=True)
    if "data-geoml-2026.zip" not in os.listdir(os.getcwd()):
        print("Has a directory called data-geoml been downloaded and placed in your working directory? If not, try re-executing this code chunk")
    else:
        print("Data download OK")

DATA_PATH = os.path.join(os.getcwd())

### Load packages

In [ ]:
if 'google.colab' in str(get_ipython()):
    !pip install mapclassify
    !pip install contextily
    !pip install pysal

import os
import numpy as np
import pandas as pd
import geopandas as gpd

# pre-processing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# models
from sklearn.neural_network import MLPRegressor

# metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error

Let's start by reading in the data as a GeoDataFrame and exploring it. 

In [ ]:
gdf = gpd.read_file(os.path.join(DATA_PATH, "data-geoml", "grid_metrics.gpkg"))
gdf = gdf.dropna()

In [ ]:
gdf.columns

In [ ]:
gdf.head()

It's important to inspect data and build up an intuition for its characteristics and patterns before developing machine learning models. Let's start by mapping the percentage tree cover data. 

In [ ]:
gdf.explore(
    column="tree_cover_pct",
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="ESRI World Imagery"
)

We can also map the embedding values - let's start by mapping the first of the embeddings.

In [ ]:
gdf.explore(
    column="emb_2020_A00_mean",
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="ESRI World Imagery"
)

#### Activity!

<details>
    <summary><strong>How can you build up an intuition as to what information is captured by the embeddings dataset?</strong></summary>

You could map different embedding elements over an area that you are familiar with and look for patterns with known characteristics of the area (e.g. which embeddings correlate with the presence of industry, cropland etc.). You could also look for correlations between embeddings and readily available data, such as environmental or census data. 
</details>

## Model development pipeline

We will use the embeddings and remote sensing inputs in a supervised learning workflow to train and evaluate models that predict urban microclimate variables. Let's create a function that encapsulates the steps to train and evaluate a machine learning model given training and test data. As we develop machine learning models and iterate through testing models, we will often repeat the same code. By placing this code in a function, we can call the function as required, avoid code repetition, and make our program more succinct. 

#### Activity!

<strong>To test your understanding of the supervised learning workflow, can you add a docstring header to summarise what the function does and what data it returns. Feel free to use an LLM to help you understand the operations inside the function and create a descriptive function summary.</strong>

<strong>Can you create a similar function that fits a simple linear regression model instead of a multi-layer perceptron deep neural network? This is the link for the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html" target="_blank">sklearn LinearRegression docs</a>.</strong>

In [ ]:
def train_eval_model_dnn(
    X_train,
    y_train,
    X_test,
    y_test,
):
    """
    CREATE A SUMMARY OF THE FUNCTION HERE AND WHAT DATA IT RETURNS
    """
    # create copies of the input data in the function scope
    X_train = X_train.copy()
    y_train = y_train.copy()
    X_test = X_test.copy()
    y_test = y_test.copy()

    # standardise data
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # fit model
    regr = MLPRegressor(
        hidden_layer_sizes=(50,), 
        random_state=4, 
        solver="sgd", 
        max_iter=500
    ).fit(X_train_scaled, y_train)

    # evaluate model
    y_test_preds = regr.predict(X_test_scaled)
    test_mse = mean_squared_error(y_test, y_test_preds)
    print(f"The MSE on the test split is: {test_mse}")
    test_r2 = r2_score(y_test, y_test_preds)
    print(f"The R2 on the test split is: {test_r2}")

    return regr, scaler

Create training and test splits with a training split consisting of a random selection of 70% of the dataset. 

**Switch which set of predictors you have commented out (ctrl or cmd - /) depending on whether you want to fit models using the embeddings or raw remote sensing data.**

**Switch the target for different urban microclimate variables that we are trying to predict.**

In [ ]:
target = "tree_cover_pct"
# predictors = [
#     "landsat_b1_mean",
#     "landsat_b2_mean",
#     "landsat_b3_mean",
#     "landsat_b4_mean",
#     "landsat_b5_mean",
#     "landsat_b6_mean",
#     "lai_202001_mean"
# ]
predictors = gdf.columns.str.match(r"emb_2020_A.*_mean")
X = gdf.loc[:, predictors]
y = gdf.loc[:, target]

# set aside 30% of the data as a test split
# set the random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=4) 

Now, we can pass the training and test data into the function we created above to train and evaluate the model

In [ ]:
model, scaler = train_eval_model_dnn(X_train, y_train, X_test, y_test,)

#### Activity!

**Can you pass the training and test data into the function you created to train linear regression models and compare the result to the deep neural network model?** You will need to create a few 

### Predictions

Now that we have a trained model, we can use it to generate predictions. We need to standardise the predictors and then pass them into the trained model's `predict()` method. This will return a vector of predictions that we can append as a column in our GeoDataFrame. 

In [ ]:
# predictors = [
#     "landsat_b1_mean",
#     "landsat_b2_mean",
#     "landsat_b3_mean",
#     "landsat_b4_mean",
#     "landsat_b5_mean",
#     "landsat_b6_mean",
#     "lai_202001_mean"
# ]
predictors = gdf.columns.str.match(r"emb_2020_A.*_mean")
X_all = gdf.loc[:, predictors]
X_all_scaled = scaler.transform(X_all)
gdf["preds"] = model.predict(X_all_scaled)

To get a feel for the spatial performance of the model, we can plot the difference between the predicted and actual tree cover (the model's prediction error). 

In [ ]:
gdf["preds_diff"] = gdf["preds"] - gdf[target]

In [ ]:
gdf.explore(
    column="preds_diff",
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="ESRI World Imagery"
)

Change the target variable and input predictors in the code above to address the following questions (these are repeated from above):

* how does a linear fit on embeddings compare to more complex models fitted on "raw" remote sensing data?
* compare and contrast the performance of the models trained with embeddings versus "raw" remote sensing data. 
* which input dataset and model would you use to generate Perth-wide maps of these variables? Consider both model performance, costs of data processing for deployment, and interpretability. 


#### Activity!

<details>
    <summary><strong>Why is mapping a model's prediction error a useful activity?</strong></summary>

You can see if the model underperformed in certain locations. This might give you clues as to how to further improve the model. For example, the model might have large errors in areas with certain characteristics which could mean you need to collect more training data from those locations. Or, the model might perform very badly in a couple of locations that is hard to expain. This could be a clue that there is some noise in the input data that should be checked.  
</details>

<br>

<details>
    <summary><strong>Why should we be cautious about assessing the model's performance using this map?</strong></summary>

70 % of these data points are in the training dataset. The model will have seen these locations during training. Therefore, it is likely the model will have lower error rates predicting on the training dataset compared to when it generalises to unseen locations.   
</details>



## Evaluating spatial predictions

Here, we created a test set that was randomly sampled from our initial dataset. This means the training and test data will be distributed evenly across the study area. This can mean the test data is representative of how the model would perform over this location. However, it can also present problems for model evaluation. Firstly, if training and test points are close together they can be spatially correlated, which means training points can contain some information about the test points. This means the test data is not independent from the training dataset and might bias model evaluations. Secondly, it does not permit an assessment of how the model would generalise to new locations. 

#### Activity!

<details>
    <summary><strong>Can you think of a strategy for evaluating the model that accounts for the spatial relationship between training and test data?</strong></summary>

We could divide our study area into zones and use a selection of zones for training and different zones for testing. This would let us assess how well the model generalises into new zones. Depending on the nature of spatial correlation in the dataset, you might consider leaving a buffer between the zones.
</details>